# PathOGen Phase 2 evaluation on Colab

Runs Workflow 06 then Workflow 07 on a T4.  The data archive is streamed into a small matched subset: images, spatial maps, and the morphology table only.  Each downloaded ZIP is deleted directly after extraction.

Use a GPU runtime.  Set `TILE_COUNT = 2000` below to run the larger evaluation.

In [ ]:
from pathlib import Path
import shutil, subprocess, sys

ROOT = Path('/content')
REPO = ROOT / 'PathOGen'
DATA_ROOT = REPO / 'data'
MODELS = REPO / 'models'
RUN_DIR = DATA_ROOT / 'evaluations' / 'phase2_1000_seed42'
TILE_COUNT = 1000  # Set to 2000 for the larger run.
SAMPLE_SEED = 42
GENERATION_STEPS = 20
DATA_FILE_ID = '1sBc4-CexT3S2cw1LZysrX4BLjVjN6BPt'
MODEL_FILE_ID = '1yc_bxgWyOD1gFeTQxUal-IGpqssQdIPd'

assert shutil.which('nvidia-smi'), 'Switch Colab to a GPU runtime first.'
print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], text=True))

In [ ]:
# Clone the exact repository revision and install all dependencies needed by Workflows 06/07.
if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', 'refactored-structure', '--single-branch',
                    'https://github.com/a12dongithub/PathOGen.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '0099c57'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'gdown'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '-e',
                '.[annotation,preprocessing,inference]'], cwd=REPO, check=True)

In [ ]:
# Download/extract models first, then delete the archive to conserve disk.
import zipfile
model_zip = ROOT / 'models.zip'
if not (MODELS / 'pathogen_phase2/checkpoint_30000').exists():
    subprocess.run([sys.executable, '-m', 'gdown', '--id', MODEL_FILE_ID, '-O', str(model_zip)], check=True)
    with zipfile.ZipFile(model_zip) as zf:
        zf.extractall(REPO)
    model_zip.unlink()
assert (MODELS / 'pathogen_phase2/checkpoint_30000').is_dir()
assert (MODELS / 'cellvit_plus_plus/cellvit_sam_h_x40_amp_001/model.pth').is_file()

In [ ]:
# Select exact aligned members directly from the data ZIP: no full-dataset extraction.
# The same sorted/sample-seed selection is used as Workflow 06.
import random, zipfile
data_zip = ROOT / 'data.zip'
if not (DATA_ROOT / 'images').exists():
    subprocess.run([sys.executable, '-m', 'gdown', '--id', DATA_FILE_ID, '-O', str(data_zip)], check=True)
    with zipfile.ZipFile(data_zip) as zf:
        names = [n for n in zf.namelist() if not n.endswith('/')]
        image_by_stem = {Path(n).stem: n for n in names if '/images/' in n and n.lower().endswith('.png')}
        map_by_stem = {Path(n).stem: n for n in names if '/spatial_maps/' in n and n.lower().endswith('.npz')}
        stems = sorted(set(image_by_stem) & set(map_by_stem))
        selected = random.Random(SAMPLE_SEED).sample(stems, min(TILE_COUNT, len(stems)))
        morphology = [n for n in names if n.endswith('/morphology_stats.parquet')]
        if len(morphology) != 1:
            raise RuntimeError(f'Expected one morphology_stats.parquet, found {morphology}')
        for d in ('images', 'spatial_maps'):
            (DATA_ROOT / d).mkdir(parents=True, exist_ok=True)
        for stem in selected:
            for member, dest in ((image_by_stem[stem], DATA_ROOT / 'images' / f'{stem}.png'),
                                 (map_by_stem[stem], DATA_ROOT / 'spatial_maps' / f'{stem}.npz')):
                with zf.open(member) as src, dest.open('wb') as out:
                    shutil.copyfileobj(src, out)
        with zf.open(morphology[0]) as src, (DATA_ROOT / 'morphology_stats.parquet').open('wb') as out:
            shutil.copyfileobj(src, out)
    data_zip.unlink()
print('subset:', len(list((DATA_ROOT / 'images').glob('*.png'))), 'tiles')

In [ ]:
# Workflow 06: generate matched images and FID/KID.  Batch size 1 is safest on a T4.
RUN_DIR = DATA_ROOT / 'evaluations' / f'phase2_{TILE_COUNT}_seed{SAMPLE_SEED}'
subprocess.run([sys.executable, 'workflows/06_evaluate_phase2_fid_kid/run.py',
    '--data-root', str(DATA_ROOT), '--images-dir', str(DATA_ROOT / 'images'),
    '--spatial-maps-dir', str(DATA_ROOT / 'spatial_maps'),
    '--morphology-table', str(DATA_ROOT / 'morphology_stats.parquet'),
    '--checkpoint', str(MODELS / 'pathogen_phase2/checkpoint_30000'),
    '--output-dir', str(RUN_DIR), '--num-tiles', str(TILE_COUNT),
    '--sample-seed', str(SAMPLE_SEED), '--steps', str(GENERATION_STEPS),
    '--batch-size', '1', '--device', 'cuda', '--dtype', 'float16',
    '--local-files-only'], cwd=REPO, check=True)

In [ ]:
# Workflow 07: re-annotate real/generated pairs and rank control consistency.
subprocess.run([sys.executable, 'workflows/07_rank_control_consistency/run.py',
    '--run-dir', str(RUN_DIR),
    '--annotation-model', str(MODELS / 'cellvit_plus_plus/cellvit_sam_h_x40_amp_001/model.pth'),
    '--cellvit-root', str(REPO / 'third_party/cellvit_plus_plus'),
    '--device', 'cuda', '--dtype', 'float16', '--n-jobs', '2',
    '--top-k', '50', '--top-k', '250'], cwd=REPO, check=True)

In [ ]:
# Archive only the requested deliverables.  Download this with colab-cli, then delete it remotely.
import tarfile
archive = DATA_ROOT / 'evaluations' / f'phase2_{TILE_COUNT}_results.tar.gz'
with tarfile.open(archive, 'w:gz') as tf:
    for relative in ['generated', 'real', 'manifest.json', 'fid_kid.json', 'control_consistency']:
        path = RUN_DIR / relative
        if path.exists(): tf.add(path, arcname=relative)
print(archive, archive.stat().st_size / 1e9, 'GB')
# Mac: colab download -s pathogen-workflows /content/PathOGen/data/evaluations/phase2_<N>_results.tar.gz \
#          /Users/varangrai/Documents/cellpathogen/data/evaluations/phase2_<N>_results.tar.gz